# Capítulo 3 — Balanços de Conservação

**Autor:** Jader Lugon Junior  
**Livro:** Fenômenos de Transporte: Fundamentos e Modelagem Computacional  
**Repositório:** [github.com/JaderLugon/fenomenos-transporte-livro](https://github.com/JaderLugon/fenomenos-transporte-livro)

---

## 🎯 Objetivos de Aprendizagem

Ao final deste notebook, você será capaz de:

1. **Aplicar** a analogia da conta bancária para montar balanços de propriedades
2. **Diferenciar** descrições Lagrangeana e Euleriana de escoamentos
3. **Interpretar** o Teorema do Transporte de Reynolds (TTR)
4. **Deduzir** a forma diferencial das equações de conservação
5. **Calcular** números adimensionais e classificar regimes de escoamento
6. **Identificar** a hierarquia Navier-Stokes → Euler → Saint-Venant

---

## 📚 Conteúdo Teórico Resumido

### 3.1 Analogia da Conta Bancária

Qualquer propriedade $B$ em um sistema físico evolui segundo:

$$\frac{dB_{\text{sistema}}}{dt} = \underbrace{\dot{B}_{\text{entrada}}}_{\text{depósitos}} - \underbrace{\dot{B}_{\text{saída}}}_{\text{saques}} + \underbrace{\dot{G}}_{\text{juros}} - \underbrace{\dot{C}}_{\text{taxas}}$$

### 3.2 Teorema do Transporte de Reynolds (TTR)

Conecta a descrição Lagrangeana (seguindo a partícula) à Euleriana (volume fixo):

$$\frac{DB}{Dt} = \frac{\partial}{\partial t} \int_{V_C} \rho \beta \, dV + \oint_{S_C} \rho \beta (\mathbf{u} \cdot \mathbf{n}) \, dA$$

### 3.3 Hierarquia de Modelos

| Modelo | Hipóteses | Aplicação |
|--------|-----------|-----------|
| Navier-Stokes | Fluido newtoniano, incompressível | Escoamentos gerais |
| Euler | Fluido ideal ($\mu = 0$) | Fora da camada limite |
| Saint-Venant | Águas rasas, pressão hidrostática | Rios, canais, estuários |

### 3.4 Números Adimensionais Fundamentais

| Número | Fórmula | Significado |
|--------|---------|-------------|
| Reynolds ($\mathrm{Re}$) | $UL/\nu$ | Inércia/Viscosidade |
| Péclet ($\mathrm{Pe}$) | $UL/D$ | Advecção/Difusão |
| Prandtl ($\mathrm{Pr}$) | $\nu/\alpha$ | Camada hidrodinâmica/térmica |
| Schmidt ($\mathrm{Sc}$) | $\nu/D$ | Camada hidrodinâmica/mássica |
| Froude ($\mathrm{Fr}$) | $U/\sqrt{gL}$ | Inércia/Gravidade |
| Damköhler ($\mathrm{Da}$) | $k_{rxn}L/U$ | Reação/Transporte |

---

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import sys
import os

# Adiciona o diretório raiz ao path para importar módulos
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from src import hidrodinamica, utils

# Configuração de plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✅ Ambiente configurado com sucesso!")
print(f"📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}")

---

## 🔬 Seção 1: Analogia da Conta Bancária

### Exercício 1.1: Tanque Industrial com Entrada e Saída

Um tanque recebe água a $0,8$ L/s e drena a $0,5$ L/s. Calcule o tempo para o nível subir de $0,5$ m para $1,2$ m, considerando área da base de $4$ m$^2$.

In [ ]:
# ============================================================
# EXERCÍCIO 1.1: Balanço de Massa em Tanque
# ============================================================

print("=" * 60)
print("EXERCÍCIO 1.1: BALANÇO DE MASSA EM TANQUE INDUSTRIAL")
print("=" * 60)

# Dados do problema
Q_ent = 0.8e-3  # m³/s (0,8 L/s)
Q_sai = 0.5e-3  # m³/s (0,5 L/s)
A_base = 4.0    # m²
h_ini = 0.5     # m
h_fin = 1.2     # m

# Cálculo
Q_liq = Q_ent - Q_sai  # Vazão líquida de entrada
dh_dt = Q_liq / A_base  # Taxa de variação do nível
delta_h = h_fin - h_ini
delta_t = delta_h / dh_dt

print(f"\n📊 Dados do problema:")
print(f"  • Vazão de entrada: Q_ent = {Q_ent*1000:.1f} L/s")
print(f"  • Vazão de saída: Q_sai = {Q_sai*1000:.1f} L/s")
print(f"  • Área da base: A = {A_base} m²")
print(f"  • Nível inicial: h_ini = {h_ini} m")
print(f"  • Nível final: h_fin = {h_fin} m")

print(f"\n🧮 Cálculo:")
print(f"  • Vazão líquida: Q_liq = {Q_ent*1000:.1f} - {Q_sai*1000:.1f} = {Q_liq*1000:.1f} L/s")
print(f"  • Taxa de variação: dh/dt = Q_liq/A = {Q_liq*1000:.1f}×10⁻³/{A_base} = {dh_dt:.2e} m/s")
print(f"  • Variação de nível: Δh = {h_fin} - {h_ini} = {delta_h} m")
print(f"  • Tempo: Δt = Δh/(dh/dt) = {delta_t:.0f} s")

print(f"\n💡 Resultado:")
print(f"  ⏱️ Tempo necessário: {delta_t:.0f} s ≈ {delta_t/60:.1f} min ≈ {delta_t/3600:.2f} h")

print(f"\n🔍 Interpretação:")
print(f"  O tanque está em regime TRANSIENTE (acumulando massa).")
print(f"  Se Q_ent = Q_sai, o sistema estaria em regime PERMANENTE.")

---

## 🔢 Seção 2: Números Adimensionais

### Exercício 2.1: Calculadora de Números Adimensionais

Calcule $\mathrm{Re}$, $\mathrm{Pe}$, $\mathrm{Pr}$ e $\mathrm{Sc}$ para diferentes cenários.

In [ ]:
# ============================================================
# EXERCÍCIO 2.1: Calculadora de Números Adimensionais
# ============================================================

def calcular_reynolds(U, L, nu):
    """Calcula o número de Reynolds"""
    return U * L / nu

def calcular_peclet(U, L, D):
    """Calcula o número de Péclet"""
    return U * L / D

def calcular_prandtl(nu, alpha):
    """Calcula o número de Prandtl"""
    return nu / alpha

def calcular_schmidt(nu, D):
    """Calcula o número de Schmidt"""
    return nu / D

def calcular_froude(U, L, g=9.81):
    """Calcula o número de Froude"""
    return U / np.sqrt(g * L)

def classificar_regime(Re, Re_lam=2000, Re_turb=2400):
    """Classifica o regime de escoamento"""
    if Re < Re_lam:
        return "LAMINAR"
    elif Re > Re_turb:
        return "TURBULENTO"
    else:
        return "TRANSIÇÃO"

def classificar_transporte(Pe):
    """Classifica o regime de transporte"""
    if Pe > 10:
        return "ADVECÇÃO DOMINANTE"
    elif Pe < 0.1:
        return "DIFUSÃO DOMINANTE"
    else:
        return "MISTO (advecção ≈ difusão)"

# Cenário 1: Água em tubo residencial
print("=" * 60)
print("CENÁRIO 1: ÁGUA EM TUBO RESIDENCIAL")
print("=" * 60)

U = 1.0        # m/s
D_tubo = 0.025 # m (25 mm)
nu_agua = 1.0e-6   # m²/s
alpha_agua = 1.4e-7  # m²/s
D_sal = 1.5e-9   # m²/s (difusividade do sal em água)

Re = calcular_reynolds(U, D_tubo, nu_agua)
Pe_termico = calcular_peclet(U, D_tubo, alpha_agua)
Pe_massico = calcular_peclet(U, D_tubo, D_sal)
Pr = calcular_prandtl(nu_agua, alpha_agua)
Sc = calcular_schmidt(nu_agua, D_sal)

print(f"\n📊 Parâmetros:")
print(f"  • Velocidade: U = {U} m/s")
print(f"  • Diâmetro: D = {D_tubo*1000} mm")
print(f"  • ν (água) = {nu_agua:.2e} m²/s")

print(f"\n🔢 Números Adimensionais:")
print(f"  • Re = {Re:.0f} → {classificar_regime(Re)}")
print(f"  • Pe_térmico = {Pe_termico:.0f} → {classificar_transporte(Pe_termico)}")
print(f"  • Pe_mássico = {Pe_massico:.0f} → {classificar_transporte(Pe_massico)}")
print(f"  • Pr = {Pr:.2f}")
print(f"  • Sc = {Sc:.0f}")

print(f"\n💡 Interpretação:")
print(f"  • Pr > 1 → camada limite térmica mais fina que a hidrodinâmica")
print(f"  • Sc >> 1 → sal difunde MUITO mais devagar que o momentum")
print(f"  • Por isso corantes demoram tanto para se misturar em água parada!")

# Cenário 2: Rio Macaé
print("\n" + "=" * 60)
print("CENÁRIO 2: RIO MACAÉ (RJ)")
print("=" * 60)

U_rio = 0.20     # m/s
L_rio = 42.2     # m (largura)
E_L = 12.57      # m²/s (dispersão longitudinal)
E_T = 0.00431    # m²/s (dispersão transversal)
D_prof = 0.71    # m (profundidade)

Re_rio = calcular_reynolds(U_rio, D_prof, nu_agua)
Pe_L = calcular_peclet(U_rio, L_rio, E_L)
Pe_T = calcular_peclet(U_rio, L_rio, E_T)
Fr_rio = calcular_froude(U_rio, D_prof)

print(f"\n📊 Parâmetros:")
print(f"  • Velocidade: U = {U_rio} m/s")
print(f"  • Largura: B = {L_rio} m")
print(f"  • Profundidade: D = {D_prof} m")
print(f"  • E_L = {E_L} m²/s | E_T = {E_T} m²/s")

print(f"\n🔢 Números Adimensionais:")
print(f"  • Re = {Re_rio:.0f} → {classificar_regime(Re_rio)}")
print(f"  • Pe_L (longitudinal) = {Pe_L:.2f} → {classificar_transporte(Pe_L)}")
print(f"  • Pe_T (transversal) = {Pe_T:.0f} → {classificar_transporte(Pe_T)}")
print(f"  • Fr = {Fr_rio:.3f} → {'SUBCRÍTICO' if Fr_rio < 1 else 'SUPERCRÍTICO'}")

print(f"\n💡 Interpretação:")
print(f"  • Pe_L ≈ 1 → advecção e dispersão longitudinal são comparáveis")
print(f"  • Pe_T >> 1 → dispersão transversal é muito lenta")
print(f"  • Fr < 1 → regime fluvial (ondas podem subir o rio)")
print(f"  • Por isso plumas de poluentes se alongam antes de se misturar!")

---

## 📊 Seção 3: Visualização de Regimes

Vamos criar um diagrama de regimes baseado em Re e Pe.

In [ ]:
# ============================================================
# SEÇÃO 3: DIAGRAMA DE REGIMES
# ============================================================

fig, ax = plt.subplots(figsize=(10, 7))

# Regiões de regime
# Pe < 1 (difusão dominante)
ax.axhspan(0, 1, alpha=0.2, color='blue', label='Difusão dominante (Pe < 1)')
# Pe > 1 (advecção dominante)
ax.axhspan(1, 1e8, alpha=0.2, color='green', label='Advecção dominante (Pe > 1)')

# Linhas de transição de Re
ax.axvline(x=2000, color='orange', linestyle='--', linewidth=2, label='Re = 2000 (laminar)')
ax.axvline(x=2400, color='red', linestyle='--', linewidth=2, label='Re = 2400 (turbulento)')

# Linha Pe = 1
ax.axhline(y=1, color='purple', linestyle='-', linewidth=2, label='Pe = 1')

# Pontos de exemplo
# Microcanal: baixo Re, baixo Pe
ax.plot(10, 0.01, 'ko', markersize=10, label='Microcanal')
ax.annotate('Microcanal\n(Re=10, Pe=0.01)', xy=(10, 0.01), xytext=(50, 0.005),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

# Tubulação residencial: Re moderado, Pe alto
ax.plot(25000, 25000, 'ro', markersize=10, label='Tubulação')
ax.annotate('Tubulação\n(Re=25000, Pe=25000)', xy=(25000, 25000), xytext=(100000, 5000),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

# Rio: alto Re, Pe moderado
ax.plot(70000, 1.6, 'go', markersize=10, label='Rio Macaé')
ax.annotate('Rio Macaé\n(Re=70000, Pe=1.6)', xy=(70000, 1.6), xytext=(200000, 0.1),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Número de Reynolds (Re)', fontsize=12)
ax.set_ylabel('Número de Péclet (Pe)', fontsize=12)
ax.set_title('Diagrama de Regimes: Re vs Pe', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim([1, 1e8])
ax.set_ylim([0.001, 1e8])

plt.tight_layout()
plt.show()

print("\n💡 Interpretação do diagrama:")
print("   • Quadrante inferior esquerdo: difusão + laminar (microcanais)")
print("   • Quadrante superior direito: advecção + turbulento (rios, tubulações)")
print("   • A linha Pe = 1 separa regimes dominados por advecção vs difusão")
print("   • As linhas Re = 2000 e 2400 separam laminar, transição e turbulento")

---

## 🧮 Seção 4: Adimensionalização da Equação de Advecção-Difusão

### Exercício 4.1: Derivação do Número de Péclet

Adimensionalize a equação de advecção-difusão 1D:

$$\frac{\partial C}{\partial t} + u\frac{\partial C}{\partial x} = E\frac{\partial^2 C}{\partial x^2}$$

In [ ]:
# ============================================================
# EXERCÍCIO 4.1: Adimensionalização
# ============================================================

print("=" * 60)
print("EXERCÍCIO 4.1: ADIMENSIONALIZAÇÃO DA EQUAÇÃO DE TRANSPORTE")
print("=" * 60)

print("\n📝 Equação dimensional:")
print("   ∂C/∂t + u·∂C/∂x = E·∂²C/∂x²")

print("\n🔄 Escalas características:")
print("   • Comprimento: x* = x/L")
print("   • Tempo: t* = t/(L/U)")
print("   • Velocidade: u* = u/U")
print("   • Concentração: C* = C/C₀")

print("\n🧮 Substituindo na equação:")
print("   (C₀·U/L)·∂C*/∂t* + (U·C₀/L)·u*·∂C*/∂x* = (E·C₀/L²)·∂²C*/∂x*²")

print("\n✂️ Dividindo por (U·C₀/L):")
print("   ∂C*/∂t* + u*·∂C*/∂x* = (E/(U·L))·∂²C*/∂x*²")

print("\n🎯 Identificando o número adimensional:")
print("   ∂C*/∂t* + u*·∂C*/∂x* = (1/Pe)·∂²C*/∂x*²")
print("   onde Pe = U·L/E é o número de Péclet")

print("\n💡 Limites assintóticos:")
print("   • Pe → 0 (difusão dominante): ∂C*/∂t* = ∂²C*/∂x*² (equação de difusão pura)")
print("   • Pe → ∞ (advecção dominante): ∂C*/∂t* + u*·∂C*/∂x* = 0 (advecção pura)")

print("\n🔍 Interpretação:")
print("   O número de Péclet surge NATURALMENTE da adimensionalização.")
print("   Ele dita quais termos da equação podem ser desprezados.")

---

## 🌊 Seção 5: Hierarquia de Modelos

### Exercício 5.1: Interpretação de Regimes em Estuário

Em um estuário, $\mathrm{Re} \approx 10^5$, $\mathrm{Fr} \approx 0,3$ e $\mathrm{Da} \approx 2$. Interprete fisicamente cada regime.

In [ ]:
# ============================================================
# EXERCÍCIO 5.1: Interpretação de Regimes
# ============================================================

print("=" * 60)
print("EXERCÍCIO 5.1: INTERPRETAÇÃO DE REGIMES EM ESTUÁRIO")
print("=" * 60)

# Dados
Re_est = 1e5
Fr_est = 0.3
Da_est = 2.0

print(f"\n📊 Números adimensionais:")
print(f"  • Re = {Re_est:.0e}")
print(f"  • Fr = {Fr_est}")
print(f"  • Da = {Da_est}")

print(f"\n🔍 Interpretação física:")
print(f"\n  1️⃣ Reynolds (Re = 10⁵ >> 1):")
print(f"     • Forças de inércia DOMINAM sobre forças viscosas")
print(f"     • Escoamento ALTAMENTE TURBULENTO")
print(f"     • Difusão molecular é desprezível")
print(f"     • Transporte governado por difusão turbulenta (ν_t, E_t)")

print(f"\n  2️⃣ Froude (Fr = 0,3 < 1):")
print(f"     • Forças gravitacionais DOMINAM sobre inércia")
print(f"     • Escoamento FLUVIAL (subcrítico)")
print(f"     • Velocidade < velocidade de propagação de ondas (U < √(g·h))")
print(f"     • Perturbações a jusante podem se propagar para montante")
print(f"     • Hipótese de PRESSÃO HIDROSTÁTICA é válida")
print(f"     • Permite uso das equações de Saint-Venant (águas rasas)")

print(f"\n  3️⃣ Damköhler (Da = 2 > 1):")
print(f"     • Tempo de reação < tempo de transporte")
print(f"     • Regime de CONTROLE CINÉTICO MODERADO")
print(f"     • Reação química/biológica ocorre rápido o suficiente")
print(f"     • Altera significativamente o perfil de concentração")
print(f"     • NÃO pode ser tratada como equilíbrio instantâneo")
print(f"     • Requer cinética explícita (ex: 1ª ordem, Michaelis-Menten)")

print(f"\n📐 Equação governante proposta:")
print(f"   ∂(hC)/∂t + ∇·(h·u·C) = ∇·(h·E_t·∇C) + h·R(C)")
print(f"   onde:")
print(f"   • h = lâmina d'água (de Saint-Venant)")
print(f"   • u = vetor velocidade média (de Saint-Venant)")
print(f"   • E_t = tensor de dispersão turbulenta")
print(f"   • R(C) = termo de reação (ex: -k·C para 1ª ordem)")

print(f"\n💡 Síntese:")
print(f"   O estuário é modelado por:")
print(f"   • Hidrodinâmica: Saint-Venant 2D (Fr < 1, Re >> 1)")
print(f"   • Transporte: Advecção-dispersão turbulenta")
print(f"   • Reação: Cinética explícita (Da ≈ 2)")

---

## 🔬 Estudos de Caso

Os estudos de caso aplicam os conceitos deste capítulo em problemas reais de engenharia.
Clique nos links abaixo para abrir os notebooks interativos:

| Estudo de Caso | Descrição | Link |
|----------------|-----------|------|
| **Caso 3.1** | Balanço de Massa em Reator CSTR | [🔗 Abrir](../casos/03_1_Balanco_Massa_Reator_CSTR.ipynb) |
| **Caso 3.2** | Calculadora de Números Adimensionais | [🔗 Abrir](../casos/03_2_Calculadora_Numeros_Adimensionais.ipynb) |

> 💡 **Dica:** Os estudos de caso podem ser executados independentemente deste notebook principal.

---

## 📖 Referências

- BIRD, R. B.; STEWART, W. E.; LIGHTFOOT, E. N. *Transport Phenomena*. 2ª ed. Wiley, 2002.
- WHITE, F. M. *Fluid Mechanics*. 7ª ed. McGraw-Hill, 2011.
- FOX, R. W.; McDONALD, A. T.; PRITCHARD, P. J. *Introduction to Fluid Mechanics*. 8ª ed. Wiley, 2011.

---

## 🔄 Navegação

- [📚 Capítulo Anterior: Fundamentos dos Fluidos](02_Fundamentos_Fluidos.ipynb)
- [📚 Próximo Capítulo: Escoamento em Tubulações](04_Escoamento_Tubulacoes.ipynb)
- [📂 Outros Capítulos](../notebooks/)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**QR Code do Capítulo 3**  
Aponte seu celular para o QR Code no livro para acessar este notebook!

</div>

In [ ]:
print("=" * 60)
print("✅ NOTEBOOK CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print("\n🎓 Você completou o Capítulo 3!")
print("📖 Próximo passo: Capítulo 4 - Escoamento em Tubulações")
print("\n💡 Dica: Execute este notebook quantas vezes precisar!")
print("   Modifique os parâmetros e explore diferentes cenários.")